Handling of images once they have been downloaded with planet_download.ipynb up to a product allowing us to train a model. 

Replaces the run_script_AF.py once created but not needed to re-run everything. 

Will apply the extraction from the zip file of downloaded images from Planet; Prepare the images creating the PNG images from the tif file allowing for the manual annotation; and will segment the images describe them for check, and cull them to suit the model requirements, and finally apply the model training. 

In [ ]:
import sys
import os
import yaml

# Add the project root to sys.path (adjust as needed)
sys.path.append(os.path.abspath("counting_waterholes"))


import counting_boats.boat_utils.planet_utils as planet_utils
import counting_boats.boat_utils.testing as testing

Extraction of the file composite.tif obtained from the planet order and downloaded into the zip file. Renders a tif file and renames it with the date_aoi.tif outside the zip file. 

In [ ]:
import os
import yaml

import counting_boats.boat_utils.planet_utils

# Define the path to your zip file
zip_path = "images/raw_images"


# Run extraction
counting_boats.boat_utils.planet_utils.extract_zip(zip_path)

Prepare the raw tif image into a usable png in future steps. Creates a padded png image to exactly match the size dividable by the stride and tile size. 
Need to define the config to make sure it matches my paths and running the tif to png transformation. 

In [ ]:
import os
import yaml

import counting_boats.train

# #cfg config:
# with open("config_train_GPU.yaml", "r") as ymlfile:
#     cfg = yaml.load(ymlfile, Loader=yaml.FullLoader)
#     os.makedirs(cfg["output_dir"], exist_ok=True)
#     cfg["tif_dir"] = cfg.get(
#         "tif_dir", os.path.join(cfg["proj_root"], "images", "RawImages")
#     )  # This is generated so not included in the config file



#Run preparation of the tif files into png and renamed the tif. 
#prepare(r"C:\Users\adria\OneDrive - AdrianoFossati\Documents\MASTER Australia\RA\Waterholes_project\counting_waterholes\images\RawImages", cfg)
#Use relative paths not absolute 
counting_boats.train.prepare("config_train_GPU.yaml")
 

Once the png is created, as we are in the training of the model phase, I need to go on LabelMe (called here in the terminal) and manually annotate the waterholes which creates in the end a json file with all my bounding boxes. Will be needed now to segment the image and the corresponding labels. 

Once the manual annotation is done, run the segmentation of the created png image with padding. Allows for future . 

In [ ]:
import os
import yaml

import counting_boats.train

#from counting_boats.train import segment


#segment the png images
counting_boats.train.segment("config_train_GPU.yaml", train_val_split=0.8)


After segmentation, we evaluate the results of the segmentation and production of material to train the model using the "train.describe" function. 
Run the bellow cell to describe from created paths of segmented images. 

In [ ]:
import sys
import os
import yaml

import counting_boats.train

#describe the created segmented images: 
counting_boats.train.describe("config_train_GPU.yaml")


Apply the cull command which will remove images with no labels until 10% of the training set has no labels. This has to be done post segmentation as we don't know prior the the amount (depends on the segmentation). 

In [ ]:
import sys
import os
import yaml

import counting_boats.train

#describe the created segmented images: 
counting_boats.train.cull("config_train_GPU.yaml")

Config path: .\training\images
-------------------------------------------
| Training dataset statistics             |
-------------------------------------------
| Number of original images: 2            |
| Number of tiles: 2069                  |
| Number of labels: 2069                  |
| Number of individual labels                |
|   - Total: 27480                     |
|   - Per class:             |
|       - Class 0: 11340 (41.27%)        |
|       - Class 2: 9705 (35.32%)        |
|       - Class 3: 1215 (4.42%)        |
|       - Class 1: 5220 (19.00%)        |
| Background Images: 11805 (570.57%)      |
-------------------------------------------


Once the cull function is applied reducing the no instance images amount to 10%, we can try to train the model: 

In [ ]:
import sys
import os
import yaml

import counting_boats.train

#describe the created segmented images: 
counting_boats.train.train("config_train_GPU.yaml")

Training of the model when ordered directly into the cmd panel and not via notebook: 
Need to figure out how to do it from here to streamline the whole process though...

In [ ]:
python train.py --workers 2 --img 416 --batch 8 --epochs 150 --data config_train_GPU_yolo.yaml --weights yolov5s.pt --cache disk

End of this script. 